# Uber & Lyft Cab Price Regression

**Dataset:** [Uber & Lyft Cab Prices](https://www.kaggle.com/datasets/ravi72munde/uber-lyft-cab-prices)  
**Goal:** Predict the price of a cab ride based on ride attributes and weather conditions.

## Steps
1. Load & explore the data
2. Clean & preprocess
3. Feature engineering
4. Train/test split
5. Train models (Linear Regression, Random Forest)
6. Evaluate & compare

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_theme(style='whitegrid')

## 1. Load Data

Reads `cab_rides.csv` and `weather.csv` directly from the local `archive.zip`.

In [ ]:
import zipfile

with zipfile.ZipFile('archive.zip', 'r') as z:
    with z.open('cab_rides.csv') as f:
        rides = pd.read_csv(f)
    with z.open('weather.csv') as f:
        weather = pd.read_csv(f)

rides = rides.sample(n=20_000, random_state=42)

print('Rides shape:', rides.shape)
print('Weather shape:', weather.shape)
rides.head()

In [ ]:
rides.info()

In [ ]:
weather.head()

## 2. Exploratory Data Analysis

In [ ]:
print('Missing values in rides:')
print(rides.isnull().sum())
print('\nMissing values in weather:')
print(weather.isnull().sum())

In [ ]:
rides['price'].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rides['price'].hist(bins=50, ax=axes[0])
axes[0].set_title('Price Distribution')
axes[0].set_xlabel('Price ($)')

rides.groupby('cab_type')['price'].mean().plot(kind='bar', ax=axes[1], color=['#009DE0', '#FF00BF'])
axes[1].set_title('Average Price by Cab Type')
axes[1].set_xlabel('Cab Type')
axes[1].set_ylabel('Avg Price ($)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
rides.groupby('name')['price'].mean().sort_values().plot(kind='barh')
plt.title('Average Price by Service Name')
plt.xlabel('Avg Price ($)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure()
plt.scatter(rides['distance'], rides['price'], alpha=0.1, s=5)
plt.xlabel('Distance (miles)')
plt.ylabel('Price ($)')
plt.title('Price vs Distance')
plt.show()

## 3. Preprocessing & Feature Engineering

In [ ]:
rides.dropna(subset=['price'], inplace=True)

rides['datetime'] = pd.to_datetime(rides['time_stamp'], unit='ms')
rides['hour'] = rides['datetime'].dt.hour
rides['day_of_week'] = rides['datetime'].dt.dayofweek
rides['month'] = rides['datetime'].dt.month

rides['time_stamp_hr'] = (rides['time_stamp'] // 3600000) * 3600
weather['time_stamp_hr'] = (weather['time_stamp'].astype(int) // 3600) * 3600

df = rides.merge(
    weather[['temp', 'clouds', 'pressure', 'rain', 'humidity', 'wind', 'location', 'time_stamp_hr']],
    left_on=['source', 'time_stamp_hr'],
    right_on=['location', 'time_stamp_hr'],
    how='left'
)

matched = df['temp'].notna().sum()
print(f'Merged shape: {df.shape}')
print(f'Weather matched: {matched:,} / {len(df):,} rows ({matched/len(df)*100:.1f}%)')

In [ ]:
for col in ['temp', 'clouds', 'pressure', 'rain', 'humidity', 'wind']:
    df[col].fillna(df[col].median(), inplace=True)

cat_cols = ['cab_type', 'name', 'source', 'destination']
le = LabelEncoder()
for col in cat_cols:
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))

features = [
    'distance', 'surge_multiplier', 'hour', 'day_of_week', 'month',
    'temp', 'clouds', 'pressure', 'rain', 'humidity', 'wind',
    'cab_type_enc', 'name_enc', 'source_enc', 'destination_enc'
]

X = df[features].copy()
y = df['price']

X = X.fillna(X.median(numeric_only=True)).fillna(0)

print('Feature matrix shape:', X.shape)
print('Remaining NaNs:', X.isna().sum().sum())

## 4. Correlation Heatmap

In [ ]:
plt.figure(figsize=(12, 8))
corr = X.join(y).corr()
sns.heatmap(corr[['price']].sort_values('price', ascending=False), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Feature Correlations with Price')
plt.tight_layout()
plt.show()

## 5. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')

## 6. Train Models

In [ ]:
def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    mae  = mean_absolute_error(y_te, preds)
    rmse = mean_squared_error(y_te, preds) ** 0.5
    r2   = r2_score(y_te, preds)
    print(f'{name:<30}  MAE={mae:.2f}  RMSE={rmse:.2f}  R²={r2:.4f}')
    return preds, r2

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

preds_lr, r2_lr = evaluate('Linear Regression', LinearRegression(), X_train_sc, y_train, X_test_sc, y_test)

rf = RandomForestRegressor(n_estimators=30, max_depth=10, n_jobs=-1, random_state=42)
preds_rf, r2_rf = evaluate('Random Forest', rf, X_train, y_train, X_test, y_test)

## 7. Visualize Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, preds_lr, alpha=0.2, s=5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[0].set_xlabel('Actual Price ($)')
axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title('Linear Regression — Actual vs Predicted')

axes[1].scatter(y_test, preds_rf, alpha=0.2, s=5, color='orange')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[1].set_xlabel('Actual Price ($)')
axes[1].set_ylabel('Predicted Price ($)')
axes[1].set_title('Random Forest — Actual vs Predicted')

plt.tight_layout()
plt.show()

In [ ]:
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=True)

plt.figure(figsize=(8, 6))
importances.plot(kind='barh')
plt.title('Random Forest — Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 8. Summary

| Model | Key Strength |
|---|---|
| Linear Regression | Fast, interpretable baseline — assumes linear relationships |
| Random Forest | Handles non-linearity, provides feature importance |

**Key predictors** are `name` (service tier), `distance`, and `surge_multiplier`.

**Note on weather:** Rain and other weather features have minimal direct impact on price because `surge_multiplier` already captures demand-driven price changes that weather causes. Rain → higher surge → higher price, but with surge in the model, rain adds no extra signal.